In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SakilaAnalysis").getOrCreate()

# Load all necessary tables
actor = spark.read.csv("actor.csv", header=True, inferSchema=True)
category = spark.read.csv("category.csv", header=True, inferSchema=True)
film = spark.read.csv("film.csv", header=True, inferSchema=True)
film_actor = spark.read.csv("film_actor.csv", header=True, inferSchema=True)
film_category = spark.read.csv("film_category.csv", header=True, inferSchema=True)
inventory = spark.read.csv("inventory.csv", header=True, inferSchema=True)
rental = spark.read.csv("rental.csv", header=True, inferSchema=True)
payment = spark.read.csv("payment.csv", header=True, inferSchema=True)
customer = spark.read.csv("customer.csv", header=True, inferSchema=True)
address = spark.read.csv("address.csv", header=True, inferSchema=True)
city = spark.read.csv("city.csv", header=True, inferSchema=True)

In [0]:
film_category.join(category, "category_id") \
    .groupBy("name") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

In [0]:
film_category.join(category, "category_id") \
    .groupBy("name") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

In [0]:
# Join actor -> film_actor -> inventory -> rental
actor_rentals = actor.join(film_actor, "actor_id") \
    .join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .groupBy("actor_id", "first_name", "last_name") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10)

actor_rentals.show()

In [0]:
# Join category -> film_category -> inventory -> rental -> payment
category.join(film_category, "category_id") \
    .join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .join(payment, "rental_id") \
    .groupBy("name") \
    .agg(F.sum("amount").alias("total_revenue")) \
    .orderBy(F.desc("total_revenue")) \
    .limit(1) \
    .show()

In [0]:
# Join category -> film_category -> inventory -> rental -> payment
category.join(film_category, "category_id") \
    .join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .join(payment, "rental_id") \
    .groupBy("name") \
    .agg(F.sum("amount").alias("total_revenue")) \
    .orderBy(F.desc("total_revenue")) \
    .limit(1) \
    .show()

In [0]:
# Left anti join finds rows in 'film' that have no match in 'inventory'
film.join(inventory, "film_id", "left_anti") \
    .select("title") \
    .show()

In [0]:
children_actors = actor.join(film_actor, "actor_id") \
    .join(film_category, "film_id") \
    .join(category, "category_id") \
    .filter(category.name == "Children") \
    .groupBy("actor_id", "first_name", "last_name") \
    .count()

# Use Window function to handle ties for the top 3 ranks
windowSpec = Window.orderBy(F.desc("count"))
children_actors.withColumn("rank", F.dense_rank().over(windowSpec)) \
    .filter(F.col("rank") <= 3) \
    .show()

In [0]:
city_activity = customer.join(address, "address_id") \
    .join(city, "city_id") \
    .groupBy("city") \
    .agg(
        F.sum(F.when(F.col("active") == 1, 1).otherwise(0)).alias("active_count"),
        F.sum(F.when(F.col("active") == 0, 1).otherwise(0)).alias("inactive_count")
    ) \
    .orderBy(F.desc("inactive_count"))

city_activity.show()

In [0]:
# Helper to calculate rental hours
rental_data = rental.withColumn("rental_duration_hrs", 
    (F.unix_timestamp("return_date") - F.unix_timestamp("rental_date")) / 3600)

base_df = rental_data.join(inventory, "inventory_id") \
    .join(film_category, "film_id") \
    .join(category, "category_id") \
    .join(customer, "customer_id") \
    .join(address, "address_id") \
    .join(city, "city_id")

def get_top_cat_by_city_pattern(pattern):
    return base_df.filter(F.col("city").like(pattern)) \
        .groupBy("name") \
        .agg(F.sum("rental_duration_hrs").alias("total_hrs")) \
        .orderBy(F.desc("total_hrs")) \
        .limit(1)

print("Top category for cities starting with 'A':")
get_top_cat_by_city_pattern("A%").show()

print("Top category for cities containing '-':")
get_top_cat_by_city_pattern("%-%").show()